# Fit MAC Curves

Fitting of marginal abatement cost curves by sector and region.

# Packages and options

In [1]:
import pandas as pd
import numpy as np
from gdxtools import GdxFile
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

In [2]:
fn_in = "../data/maccurves.gdx"

Mapping for region and sector labels

In [3]:
map_region = {
    'aut': 'AT',
    'bel': 'BE',
    'bgr': 'BG',
    'cze': 'CZ',
    'deu': 'DE',
    'dnk': 'DK',
    'esp': 'ES',
    'est': 'EE',
    'fin': 'FI',
    'fra': 'FR',
    'grc': 'GR',
    'hrv': 'HR',
    'hun': 'HU',
    'irl': 'IE',
    'ita': 'IT',
    'ltu': 'LT',
    'lva': 'LV',
    'nld': 'NL',
    'pol': 'PL',
    'prt': 'PT',
    'rou': 'RO',
    'svk': 'SE',
    'svn': 'SI',
    'swe': 'SK'}


map_sector = {'AGR': 'Agriculture',
    'EIT': 'EnergieIntensive',
    'MAC': 'IndustryServices',
    'TRN': 'Transport',
    'cheat': 'PrivateHeat',
    'coa': 'CoalExtraction',
    'gas': 'GasExtraction',
    'htrn': 'PrivateTransport',
    'p_c': 'Refineries'}

# Get and reshape results

In [4]:
gdx = GdxFile(fn_in)
gdx.symbols

['r_stats',
 'r_welfare',
 'r_welfare0',
 'r_pcarb',
 'r_emissions',
 'r_carblim',
 'Merged_set_1']

Get solved scenarios

In [5]:
df_stats = gdx.get_symbol("r_stats", col_names=["scenario", "variable"]
                         ).to_frame("value").pivot_table("value", "scenario", "variable")
solved = list(df_stats[(df_stats.solvestat == 1) & (df_stats.modelstat == 1)].index)
print("Scenarios not solved: %d" % (len(df_stats) - len(solved)))

Scenarios not solved: 0


Function to infer scenario settings from scenario name

In [6]:
def get_settings(df, col="scenario"):
    """Infer settings from scenario name
    :param df: <pd.DataFrame> with data
    :param col: <string> name of column with scenario name"""
    df_ = df.copy()
    df_["region"] = df[col].map(lambda x: x.split("_")[-1])
    df_["sector"] = df[col].map(lambda x: "_".join(x.split("_")[2:-1]))
    df_["reduction"] = df[col].map(lambda x: float(x.split("_")[1]))
    return df_.drop("scenario", axis=1)

Carbon price

In [7]:
df_pcarb = gdx.get_symbol("r_pcarb", col_names=["scenario", "ets"]).to_frame("value")\
            .pivot_table("value", "scenario", "ets")["ets1"].to_frame("pcarb").reset_index()
df_pcarb = get_settings(df_pcarb)
df_pcarb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14868 entries, 0 to 14867
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   pcarb      14868 non-null  float64
 1   region     14868 non-null  object 
 2   sector     14868 non-null  object 
 3   reduction  14868 non-null  float64
dtypes: float64(2), object(2)
memory usage: 464.8+ KB


Emissions (MtCO2)

In [8]:
df_carb = gdx.get_symbol("r_carblim", col_names=["scenario", "type", "ets"]).to_frame("value").reset_index()
df_carb = (df_carb[(df_carb.ets == "ets1")].drop("ets", axis=1)\
           .pivot_table("value", "scenario", "type")).reset_index()
df_carb = get_settings(df_carb)
df_carb["abatement"] = df_carb.baseline - df_carb.imposed
df_carb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14868 entries, 0 to 14867
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   adjusted   14868 non-null  float64
 1   baseline   14868 non-null  float64
 2   imposed    14868 non-null  float64
 3   region     14868 non-null  object 
 4   sector     14868 non-null  object 
 5   reduction  14868 non-null  float64
 6   abatement  14868 non-null  float64
dtypes: float64(5), object(2)
memory usage: 813.2+ KB


All in one

In [9]:
df_all = df_pcarb.merge(df_carb, on=["region", "sector", "reduction"])
df_all

,pcarb,region,sector,reduction,adjusted,baseline,imposed,abatement
0,5.682231,REU,AGR,0.01,0.140457,0.141875,0.140457,0.001419
1,6.625478,aut,AGR,0.01,0.674291,0.681102,0.674291,0.006811
2,3.774305,bel,AGR,0.01,1.420183,1.434528,1.420183,0.014345
3,3.437129,bgr,AGR,0.01,0.469691,0.474436,0.469691,0.004744
4,3.154200,cze,AGR,0.01,1.132583,1.144023,1.132583,0.011440
...,...,...,...,...,...,...,...,...
14899,2201.597690,prt,p_c,0.60,0.798821,1.997053,0.798821,1.198232
14900,1733.203158,rou,p_c,0.60,0.899428,2.248570,0.899428,1.349142
14901,1203.742282,svk,p_c,0.60,0.713189,1.782973,0.713189,1.069784
14902,1775.659489,svn,p_c,0.60,0.000316,0.000791,0.000316,0.000475


# Fit abatement cost function

In [10]:
def fit_mac(df, region, sector, func):
    """Fitting of marginal abatement cost function
    :param df: <pd.DataFrame> with all data
    :param region: <string> region for which to fit the mac
    :param sector: <string> sector for which to fit the mac
    :param func: <function> to be fitted. First argument has to be x-value
    :return: <(tuple, float)> tuple with coefficients and bau emissions
    """
    # extract data to fit
    df = df_all.copy()
    df_ = df[(df.region == region) & (df.sector == sector)].copy().sort_values("abatement")
    # fit curve 
    coefs, _ = curve_fit(func, df_.abatement, df_.pcarb, bounds=(0, np.inf))
    # get bau emissions
    bau = df_.baseline.iloc[0]
    return coefs, bau


In [11]:
def linear(x, a):
    return a*x 

def quadratic(x, a, b):
    return a*x + b*x*x 



Create linear mac

In [12]:
def linear(x, a):
    return a*x 

res = []
for region in map_region: 
    for sector in map_sector:
        coef, emi = fit_mac(df_all, region, sector, linear)
        r = {}
        r["region"] = map_region[region]
        r["sector"] = map_sector[sector]
        r["coefficientLinear"] = coef[0]
        r["baselineEmissionsMt"] = emi
        res.append(r)

df_mac_linear = pd.DataFrame(res)
df_mac_linear

,region,sector,coefficientLinear,baselineEmissionsMt
0,AT,Agriculture,2.270053e+03,0.681102
1,AT,EnergieIntensive,1.608385e+02,5.911850
2,AT,IndustryServices,3.114150e+02,5.050427
3,AT,Transport,5.624411e+01,22.704286
4,AT,PrivateHeat,3.100106e+02,4.903078
...,...,...,...,...
211,SK,PrivateHeat,1.616295e+04,0.309365
212,SK,CoalExtraction,1.294178e+06,0.000054
213,SK,GasExtraction,5.909487e+05,0.000985
214,SK,PrivateTransport,8.351900e+02,6.643846


In [13]:
def quadratic(x, a, b):
    return a*x + b*x*x 

res = []
for region in map_region: 
    for sector in map_sector:
        coef, emi = fit_mac(df_all, region, sector, quadratic)
        r = {}
        r["region"] = map_region[region]
        r["sector"] = map_sector[sector]
        r["coefficientLinear"] = coef[0]
        r["coefficientQuadratic"] = coef[1]
        r["baselineEmissionsMt"] = emi
        res.append(r)

df_mac_quadratic = pd.DataFrame(res)
df_mac_quadratic

,region,sector,coefficientLinear,coefficientQuadratic,baselineEmissionsMt
0,AT,Agriculture,7.653260e+01,6.556199e+03,0.681102
1,AT,EnergieIntensive,6.098497e-19,5.606387e+01,5.911850
2,AT,IndustryServices,1.630709e-22,1.268217e+02,5.050427
3,AT,Transport,2.891821e+00,4.783744e+00,22.704286
4,AT,PrivateHeat,8.773199e+00,1.250728e+02,4.903078
...,...,...,...,...,...
211,SK,PrivateHeat,8.917785e-11,1.081104e+05,0.309365
212,SK,CoalExtraction,1.294178e+06,1.015346e+00,0.000054
213,SK,GasExtraction,1.121113e+05,9.892095e+08,0.000985
214,SK,PrivateTransport,7.056077e-24,2.575885e+02,6.643846


# Export to Excel and CSV

In [14]:
df_mac_linear.to_csv("../mac_linear.csv", index=False)
df_mac_linear.to_excel("../mac_linear.xlsx", index=False)
df_mac_quadratic.to_csv("../mac_quadratic.csv", index=False)
df_mac_quadratic.to_excel("../mac_quadratic.xlsx", index=False)

In [16]:
df_mac_linear.to_csv("../../../model/data/mac_linear.csv", index=False)